## 1. Setup and Dependencies

In [ ]:
!pip install langchain langchain-mistralai requests beautifulsoup4 chromadb faiss-cpu sentence-transformers ipywidgets

print("✅ All packages installed successfully!")

## 2. Mistral API Setup and Connection Test (10 minutes)

In [ ]:
from google.colab import userdata
import os
from langchain_mistralai import ChatMistralAI

# Get Mistral API key securely
# Get your free Mistral API key from: https://console.mistral.ai/
mistral_api_key = userdata.get('MISTRAL_API_KEY')
os.environ["MISTRAL_API_KEY"] = mistral_api_key

# Initialize Mistral LLM
llm = ChatMistralAI(
    model="mistral-small",
    mistral_api_key=mistral_api_key,
    temperature=0.7,
    max_tokens=1000
)

# Test connection
try:
    response = llm.invoke("Hello! Please confirm you're working and tell me your name.")
    print("✅ Connected to Mistral successfully!")
    print(f"🤖 Response: {response.content}")
except Exception as e:
    print(f"❌ Connection failed: {str(e)}")
    print("Please check your API key and try again.")

## 3. Basic Tool - Calculator (15 minutes)

In [ ]:
from langchain.tools import Tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

def safe_calculate(expression: str) -> str:
    """Safely evaluate mathematical expressions"""
    try:
        # Remove whitespace and check for allowed characters only
        expression = expression.strip()
        allowed_chars = set('0123456789+-*/()%. ')
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters in expression. Only numbers and +, -, *, /, (, ), %, . are allowed."

        # Additional safety: prevent very long expressions
        if len(expression) > 100:
            return "Error: Expression too long"

        result = eval(expression)
        return f"Result: {result}"
    except ZeroDivisionError:
        return "Error: Division by zero"
    except SyntaxError:
        return "Error: Invalid mathematical expression syntax"
    except Exception as e:
        return f"Error: {str(e)}"

# Create calculator tool
calculator_tool = Tool(
    name="calculator",
    description="Perform mathematical calculations. Input should be a mathematical expression like '2+2', '10*5', or '(15+25)/2'. Only basic math operations are supported.",
    func=safe_calculate
)

print("🧮 Calculator tool created successfully!")

# Test the calculator tool directly
test_expressions = ["2+2", "10*5", "15/3", "(10+5)*2"]
print("\n🧪 Testing calculator tool directly:")
for expr in test_expressions:
    result = calculator_tool.func(expr)
    print(f"  {expr} = {result}")

## 4. Create First Agent - Calculator Agent (15 minutes)

In [ ]:
# Get the ReAct prompt template
react_prompt = hub.pull("hwchase17/react")

# Create the calculator agent
tools = [calculator_tool]
agent = create_react_agent(llm, tools, react_prompt)
calculator_agent = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    max_execution_time=30,
    handle_parsing_errors=True,
    early_stopping_method="generate"
)

print("🤖 Calculator Agent created successfully!")

# Test the calculator agent with better error handling
print("\n🧪 Testing Calculator Agent with natural language queries:")

test_queries = [
    "What is 25 multiplied by 47?",
    "Calculate 150 divided by 6",
    "What's 15 percent of 200?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"🙋 Query: {query}")
    print('='*60)
    try:
        result = calculator_agent.invoke({"input": query})
        if 'output' in result:
            print(f"✅ Agent Response: {result['output']}")
        else:
            print(f"⚠️  Agent Response: {result}")
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        # Try simpler direct calculation
        if "25" in query and "47" in query:
            print(f"🔧 Fallback calculation: 25 * 47 = {25 * 47}")
        elif "150" in query and "6" in query:
            print(f"🔧 Fallback calculation: 150 / 6 = {150 / 6}")
        elif "15" in query and "200" in query:
            print(f"🔧 Fallback calculation: 15% of 200 = {200 * 0.15}")

## 5. Web Search Tool Implementation (20 minutes)

In [ ]:
import requests
from langchain.tools import BaseTool
from typing import Optional, Type
from pydantic import BaseModel, Field
import json

class WebSearchInput(BaseModel):
    query: str = Field(description="Search query to look up information")

class SimpleWebSearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for current information. Use this when you need recent data or facts not in your training. Input should be a clear search query."
    args_schema: Type[BaseModel] = WebSearchInput

    def _run(self, query: str) -> str:
        """Search the web using DuckDuckGo instant answers API"""
        try:
            # Clean the query
            query = query.strip().replace(" ", "+")

            # Use DuckDuckGo instant answer API (free, no key required)
            url = f"https://api.duckduckgo.com/?q={query}&format=json&no_html=1&skip_disambig=1"

            response = requests.get(url, timeout=10)
            data = response.json()

            # Try different response fields in order of preference
            if data.get('AbstractText'):
                return f"Search result for '{query}':\n{data['AbstractText']}\n\nSource: {data.get('AbstractURL', 'DuckDuckGo')}"

            elif data.get('Definition'):
                return f"Definition for '{query}':\n{data['Definition']}\n\nSource: {data.get('DefinitionURL', 'DuckDuckGo')}"

            elif data.get('RelatedTopics') and len(data['RelatedTopics']) > 0:
                # Get the first related topic
                topic = data['RelatedTopics'][0]
                if isinstance(topic, dict) and topic.get('Text'):
                    return f"Information about '{query}':\n{topic['Text']}"

            elif data.get('Answer'):
                return f"Answer for '{query}': {data['Answer']}"

            # If no good instant answer, try to get some basic info
            return f"Search completed for '{query}'. For detailed information, you may need to search more specifically or try rephrasing your query."

        except requests.exceptions.Timeout:
            return f"Search timed out for query: {query}. Please try again with a simpler query."
        except requests.exceptions.RequestException as e:
            return f"Search failed due to network error: {str(e)}"
        except Exception as e:
            return f"Search error for '{query}': {str(e)}"

    async def _arun(self, query: str) -> str:
        return self._run(query)

# Test the web search tool
web_search_tool = SimpleWebSearchTool()
print("🌐 Web Search Tool created successfully!")

# Test searches
print("\n🧪 Testing Web Search Tool:")
test_searches = [
    "Python programming language",
    "current population Tokyo",
    "photosynthesis definition"
]

for search_query in test_searches:
    print(f"\n📍 Searching: {search_query}")
    result = web_search_tool._run(search_query)
    print(f"🔍 Result: {result[:200]}...")

## 6. Research Agent Class (25 minutes)

In [ ]:
class ResearchAgent:
    """Advanced research agent that can search the web and perform calculations"""

    def __init__(self, llm):
        self.llm = llm
        self.tools = [
            web_search_tool,
            calculator_tool
        ]

        # Create the agent with both tools
        self.agent = create_react_agent(llm, self.tools, react_prompt)
        self.agent_executor = AgentExecutor(
            agent=self.agent,
            tools=self.tools,
            verbose=True,
            max_iterations=3,
            max_execution_time=30,
            handle_parsing_errors=True,
            return_intermediate_steps=True
        )

    def research(self, query: str) -> dict:
        """Conduct research on a given query"""
        try:
            result = self.agent_executor.invoke({"input": query})
            return {
                "success": True,
                "query": query,
                "answer": result['output'],
                "error": None
            }
        except Exception as e:
            return {
                "success": False,
                "query": query,
                "answer": None,
                "error": str(e)
            }

    def batch_research(self, queries: list) -> list:
        """Research multiple queries"""
        results = []
        for query in queries:
            print(f"\n🔍 Researching: {query}")
            result = self.research(query)
            results.append(result)
        return results

# Create research agent
research_agent = ResearchAgent(llm)
print("🔬 Research Agent created successfully!")

# Test the research agent
print("\n🧪 Testing Research Agent:")

research_queries = [
    "What is the current population of Japan and how has it changed over the last decade?",
    "How does machine learning work and what are its main applications?",
    "What are the environmental benefits of solar energy?"
]

for query in research_queries:
    print(f"\n{'='*80}")
    print(f"🔬 Research Query: {query}")
    print('='*80)

    result = research_agent.research(query)
    if result['success']:
        print(f"✅ Research Complete!")
        print(f"📋 Answer: {result['answer']}")
    else:
        print(f"❌ Research Failed: {result['error']}")

## 7. Specialized Customer Service Agents (25 minutes)

In [ ]:
class SpecializedAgent:
    """Base class for specialized customer service agents"""

    def __init__(self, llm, specialty: str, tools: list, system_prompt: str = None):
        self.llm = llm
        self.specialty = specialty
        self.tools = tools
        self.system_prompt = system_prompt or f"You are a helpful {specialty} specialist."

        # Create agent
        self.agent = create_react_agent(llm, tools, react_prompt)
        self.agent_executor = AgentExecutor(
            agent=self.agent,
            tools=tools,
            verbose=False,  # Reduce verbosity for specialized agents
            max_iterations=2,
            max_execution_time=20,
            handle_parsing_errors=True,
            return_intermediate_steps=False
        )

    def handle(self, query: str) -> str:
        """Handle a customer inquiry with better error handling"""
        try:
            # For simple queries, try direct LLM response first
            if len(query.split()) < 10 and not any(word in query.lower() for word in ['calculate', 'compute', 'search']):
                enhanced_query = f"As a {self.specialty} specialist, provide a helpful response to: {query}"
                result = self.llm.invoke(enhanced_query)
                return result.content

            # For complex queries, use the agent
            enhanced_query = f"As a {self.specialty} specialist, help with: {query}"
            result = self.agent_executor.invoke({"input": enhanced_query})
            return result.get('output', 'I apologize, I was unable to process your request. Please try rephrasing.')

        except Exception as e:
            return f"I apologize for the technical difficulty. For {self.specialty} assistance with '{query}', please contact our support team directly."

class TechnicalSupportAgent(SpecializedAgent):
    """Specialized agent for technical support issues"""

    def __init__(self, llm):
        system_prompt = """You are a technical support specialist. You help customers with:
        - Software bugs and errors
        - Installation and setup issues
        - Performance problems
        - Feature usage and configuration
        - Troubleshooting steps

        Always provide clear, step-by-step solutions when possible."""

        super().__init__(
            llm=llm,
            specialty="technical support",
            tools=[calculator_tool],  # Can calculate things like memory usage, etc.
            system_prompt=system_prompt
        )

class BillingSupportAgent(SpecializedAgent):
    """Specialized agent for billing and payment issues"""

    def __init__(self, llm):
        system_prompt = """You are a billing support specialist. You help customers with:
        - Payment processing issues
        - Subscription management
        - Refund requests
        - Invoice questions
        - Pricing inquiries

        Always be empathetic and provide clear next steps."""

        super().__init__(
            llm=llm,
            specialty="billing support",
            tools=[calculator_tool],  # Can calculate refunds, prorations, etc.
            system_prompt=system_prompt
        )

class GeneralSupportAgent(SpecializedAgent):
    """Specialized agent for general inquiries"""

    def __init__(self, llm):
        system_prompt = """You are a general customer support specialist. You help customers with:
        - General product information
        - Account questions
        - Policy clarifications
        - Basic how-to questions
        - Routing to other departments when needed

        Always be friendly and helpful."""

        super().__init__(
            llm=llm,
            specialty="general customer support",
            tools=[web_search_tool],  # Can search for general info
            system_prompt=system_prompt
        )

# Create specialized agents
technical_agent = TechnicalSupportAgent(llm)
billing_agent = BillingSupportAgent(llm)
general_agent = GeneralSupportAgent(llm)

print("👥 Specialized Customer Service Agents created successfully!")
print("  ⚙️ Technical Support Agent")
print("  💳 Billing Support Agent")
print("  🏢 General Support Agent")

## 8. Multi-Agent Customer Service System (20 minutes)

In [ ]:
class CustomerServiceSystem:
    """Intelligent customer service system with multiple specialized agents"""

    def __init__(self, llm):
        self.llm = llm

        # Initialize specialized agents
        self.technical_agent = TechnicalSupportAgent(llm)
        self.billing_agent = BillingSupportAgent(llm)
        self.general_agent = GeneralSupportAgent(llm)

        # Remove the router agent since we're using direct LLM calls
        # self.router_agent = create_react_agent(llm, [], prompt)
        # self.router_executor = AgentExecutor(...)

    def route_inquiry(self, inquiry: str) -> str:
        """Determine which agent should handle the inquiry using direct LLM call"""
        routing_prompt = f"""Classify this customer inquiry. Respond with exactly one word: TECHNICAL, BILLING, or GENERAL.

TECHNICAL: app crashes, bugs, errors, installation, performance, login
BILLING: payments, subscriptions, refunds, invoices, pricing
GENERAL: information, account questions, policies, how-to

Inquiry: "{inquiry}"

Classification:"""

        try:
            # Use direct LLM call instead of agent for simple classification
            result = self.llm.invoke(routing_prompt)
            classification = result.content.strip().upper()

            # Extract the classification
            if "TECHNICAL" in classification:
                return "TECHNICAL"
            elif "BILLING" in classification:
                return "BILLING"
            else:
                return "GENERAL"
        except Exception as e:
            print(f"Routing failed: {e}, defaulting to GENERAL")
            return "GENERAL"

    def handle_inquiry(self, inquiry: str) -> dict:
        """Process a customer inquiry through the appropriate agent"""
        # Route the inquiry
        route = self.route_inquiry(inquiry)

        # Handle with appropriate agent
        if route == "TECHNICAL":
            agent_name = "Technical Support"
            response = self.technical_agent.handle(inquiry)
        elif route == "BILLING":
            agent_name = "Billing Support"
            response = self.billing_agent.handle(inquiry)
        else:
            agent_name = "General Support"
            response = self.general_agent.handle(inquiry)

        return {
            "inquiry": inquiry,
            "routed_to": route,
            "agent_name": agent_name,
            "response": response
        }

    def batch_handle(self, inquiries: list) -> list:
        """Handle multiple inquiries"""
        results = []
        for inquiry in inquiries:
            result = self.handle_inquiry(inquiry)
            results.append(result)
        return results

# Create the customer service system
cs_system = CustomerServiceSystem(llm)
print("🎯 Multi-Agent Customer Service System created successfully!")

# Test the system with simpler queries
print("\n🧪 Testing Multi-Agent Customer Service System:")

test_inquiries = [
    "App crashes on startup",
    "Need refund for last month",
    "What are your hours?",
    "Password reset help"
]

for inquiry in test_inquiries:
    print(f"\n{'='*80}")
    print(f"🎫 Customer Inquiry: {inquiry}")
    print('='*80)

    result = cs_system.handle_inquiry(inquiry)
    print(f"🎯 Routed to: {result['agent_name']} ({result['routed_to']})")
    print(f"💬 Response: {result['response']}")

## 9. Interactive Demo Interface Setup (15 minutes)

In [ ]:
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# Create the main demo interface class
class AgentDemoInterface:
    """Interactive demo interface for all agents using Jupyter widgets"""

    def __init__(self):
        # Store references to all our agents
        self.calculator_agent = calculator_agent
        self.research_agent = research_agent
        self.cs_system = cs_system

        # Create the interface widgets
        self.create_widgets()
        self.setup_event_handlers()

        # Display the interface
        self.display_interface()

    def create_widgets(self):
        """Create all the interface widgets"""

        # Title
        self.title = widgets.HTML(
            value="<h2>🤖 Multi-Agent AI System Demo</h2><p>Test all the agents we've built in this workshop!</p>",
            layout=widgets.Layout(margin='0 0 20px 0')
        )

        # Agent selector
        self.agent_selector = widgets.Dropdown(
            options=[
                ('🧮 Calculator Agent', 'calculator'),
                ('🔬 Research Agent', 'research'),
                ('🎯 Customer Service System', 'customer_service')
            ],
            value='calculator',
            description='Choose Agent:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )

        # Query input
        self.query_input = widgets.Textarea(
            value='What is 25 * 47 + 138?',
            placeholder='Enter your query here...',
            description='Your Query:',
            layout=widgets.Layout(width='100%', height='100px'),
            style={'description_width': '120px'}
        )

        # Submit button
        self.submit_button = widgets.Button(
            description='🚀 Submit Query',
            button_style='primary',
            layout=widgets.Layout(width='200px', height='40px')
        )

        # Clear button
        self.clear_button = widgets.Button(
            description='🗑️ Clear Output',
            button_style='warning',
            layout=widgets.Layout(width='150px', height='40px')
        )

        # Output area
        self.output_area = widgets.Output(
            layout=widgets.Layout(
                border='1px solid #ccc',
                padding='10px',
                margin='10px 0',
                height='400px',
                overflow='auto'
            )
        )

        # Example queries for each agent
        self.examples = {
            'calculator': [
                'What is 25 * 47 + 138?',
                'Calculate 15% of 250',
                'If I have $1000 and spend $347, how much is left?'
            ],
            'research': [
                'What is the current population of Tokyo?',
                'How does solar energy work?',
                'What are the benefits of electric vehicles?'
            ],
            'customer_service': [
                'My app keeps crashing when I upload files',
                'I was charged twice for my subscription',
                'How do I change my password?'
            ]
        }

        # Example buttons
        self.example_buttons = {}
        for agent_type, examples in self.examples.items():
            buttons = []
            for i, example in enumerate(examples):
                btn = widgets.Button(
                    description=f"Example {i+1}",
                    button_style='info',
                    layout=widgets.Layout(width='100px', margin='2px')
                )
                btn.example_text = example
                btn.on_click(self.load_example)
                buttons.append(btn)
            self.example_buttons[agent_type] = buttons

    def setup_event_handlers(self):
        """Setup event handlers for widgets"""
        self.submit_button.on_click(self.handle_query)
        self.clear_button.on_click(self.clear_output)
        self.agent_selector.observe(self.on_agent_change, names='value')

    def display_interface(self):
        """Display the complete interface"""

        # Control panel
        controls = widgets.VBox([
            self.title,
            self.agent_selector,
            widgets.HTML("<b>Quick Examples:</b>"),
            widgets.HBox(self.example_buttons[self.agent_selector.value]),
            self.query_input,
            widgets.HBox([self.submit_button, self.clear_button])
        ])

        # Main interface
        main_interface = widgets.VBox([
            controls,
            widgets.HTML("<b>Output:</b>"),
            self.output_area
        ])

        display(main_interface)

    def on_agent_change(self, change):
        """Handle agent selection change"""
        new_agent = change['new']

        # Update example buttons
        for widget in self.example_buttons[new_agent]:
            widget.layout.display = 'block'

        # Update example text based on agent
        examples = self.examples[new_agent]
        if examples:
            self.query_input.value = examples[0]

    def load_example(self, button):
        """Load example query when button is clicked"""
        self.query_input.value = button.example_text

    def clear_output(self, button):
        """Clear the output area"""
        with self.output_area:
            clear_output()

    def handle_query(self, button):
        """Handle query submission"""
        with self.output_area:
            clear_output()

            agent_type = self.agent_selector.value
            query = self.query_input.value.strip()

            if not query:
                print("❌ Please enter a query")
                return

            print(f"🤖 Processing with {self.agent_selector.label} agent...")
            print(f"📝 Query: {query}")
            print("=" * 60)

            try:
                if agent_type == 'calculator':
                    result = self.calculator_agent.invoke({"input": query})
                    print(f"🧮 Calculator Result:")
                    print(f"✅ {result.get('output', 'Calculation completed')}")

                elif agent_type == 'research':
                    result = self.research_agent.research(query)
                    if result['success']:
                        print(f"🔬 Research Complete:")
                        print(f"📋 {result['answer']}")
                    else:
                        print(f"❌ Research Error: {result['error']}")

                elif agent_type == 'customer_service':
                    result = self.cs_system.handle_inquiry(query)
                    print(f"🎯 Customer Service Response:")
                    print(f"📍 Routed to: {result['agent_name']}")
                    print(f"💬 Response: {result['response']}")

                print("\n" + "=" * 60)
                print("✅ Query processing completed!")

            except Exception as e:
                print(f"❌ Unexpected error: {str(e)}")
                print("Please try again or contact support.")

print("🎨 Interactive demo interface class created successfully!")

## 10. Launch Interactive Demo (20 minutes)

In [ ]:
# Display usage instructions
print("🚀 Launching Interactive Multi-Agent Demo Interface!")
print("\nℹ️  How to use:")
print("1. Select an agent from the dropdown")
print("2. Click an example button or type your own query")
print("3. Click 'Submit Query' to see the agent in action")
print("4. Try different agents with various types of questions")
print("\n" + "="*60)

# Create and display the demo interface
demo_interface = AgentDemoInterface()

## 11. Performance Testing and Metrics (15 minutes)

In [ ]:
import time
from datetime import datetime

class AgentPerformanceTester:
    """Test and measure agent performance"""

    def __init__(self):
        self.test_results = []

    def test_agent_performance(self, agent_name: str, agent_func, test_queries: list):
        """Test an agent's performance with multiple queries"""
        print(f"\n🧪 Performance Testing: {agent_name}")
        print("=" * 50)

        results = {
            'agent_name': agent_name,
            'total_queries': len(test_queries),
            'successful_queries': 0,
            'failed_queries': 0,
            'total_time': 0,
            'average_time': 0,
            'query_results': []
        }

        for i, query in enumerate(test_queries, 1):
            print(f"\n📝 Test {i}/{len(test_queries)}: {query[:50]}...")

            start_time = time.time()
            try:
                result = agent_func(query)
                end_time = time.time()
                execution_time = end_time - start_time

                results['successful_queries'] += 1
                results['total_time'] += execution_time

                query_result = {
                    'query': query,
                    'success': True,
                    'execution_time': execution_time,
                    'result_length': len(str(result))
                }

                print(f"   ✅ Success ({execution_time:.2f}s)")

            except Exception as e:
                end_time = time.time()
                execution_time = end_time - start_time

                results['failed_queries'] += 1
                results['total_time'] += execution_time

                query_result = {
                    'query': query,
                    'success': False,
                    'execution_time': execution_time,
                    'error': str(e)
                }

                print(f"   ❌ Failed ({execution_time:.2f}s): {str(e)[:50]}...")

            results['query_results'].append(query_result)

        # Calculate metrics
        if results['total_queries'] > 0:
            results['average_time'] = results['total_time'] / results['total_queries']
            results['success_rate'] = results['successful_queries'] / results['total_queries']

        self.test_results.append(results)
        self.print_performance_summary(results)

        return results

    def print_performance_summary(self, results):
        """Print performance summary"""
        print(f"\n📊 Performance Summary for {results['agent_name']}:")
        print(f"   Total Queries: {results['total_queries']}")
        print(f"   Successful: {results['successful_queries']}")
        print(f"   Failed: {results['failed_queries']}")
        print(f"   Success Rate: {results.get('success_rate', 0):.1%}")
        print(f"   Average Time: {results['average_time']:.2f}s")
        print(f"   Total Time: {results['total_time']:.2f}s")

# Create performance tester
tester = AgentPerformanceTester()

# Test each agent
print("🔬 Starting comprehensive agent performance testing...")

# Test Calculator Agent
calc_queries = [
    "What is 15 + 25?",
    "Calculate 12 * 8",
    "What's 100 / 4?"
]

def test_calculator(query):
    return calculator_agent.invoke({"input": query})['output']

tester.test_agent_performance("Calculator Agent", test_calculator, calc_queries)

# Test Research Agent
research_queries = [
    "What is artificial intelligence?",
    "Benefits of renewable energy"
]

def test_research(query):
    result = research_agent.research(query)
    if result['success']:
        return result['answer']
    else:
        raise Exception(result['error'])

tester.test_agent_performance("Research Agent", test_research, research_queries)

# Test Customer Service System
cs_queries = [
    "App crashes on startup",
    "Need refund help"
]

def test_customer_service(query):
    result = cs_system.handle_inquiry(query)
    return result['response']

tester.test_agent_performance("Customer Service", test_customer_service, cs_queries)
